In [10]:
import pandas as pd
import os
import numpy as np

In [11]:
# ==============================================================================
# BƯỚC 0: TẢI VÀ KIỂM TRA DỮ LIỆU GỐC (Load & Validate Raw Data)
# ==============================================================================

# 1. Khai báo đường dẫn file / xuất file
# (Luôn gán đường dẫn vào một biến để dễ dàng thay đổi ở một nơi duy nhất)
input_path = "../data/processing/data_processing.csv"

# 2. Kiểm tra sự tồn tại của file trước khi đọc
if os.path.exists(input_path):
    print(f"[THÀNH CÔNG] Đã tìm thấy file dữ liệu tại: {input_path}")
    
    # Đọc file CSV và lưu vào biến df_raw
    df_raw = pd.read_csv(input_path)
    
    # In ra báo cáo nhanh về số lượng dữ liệu
    print(f"Kích thước dữ liệu gốc: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột.\n")
    
    # Xem trước 5 dòng đầu tiên để đối chiếu định dạng các cột
    display(df_raw.head(5)) 

else:
    # Cảnh báo rõ ràng giúp người dùng biết chính xác lỗi ở đâu thay vì để Python văng lỗi đỏ lòm
    print(f"[LỖI TỚI MÁY!] Không tìm thấy file dữ liệu tại đường dẫn: '{file_path}'")
    print("Vui lòng kiểm tra lại xem file đã được đặt đúng thư mục '../data/' chưa.")

[THÀNH CÔNG] Đã tìm thấy file dữ liệu tại: ../data/processing/data_processing.csv
Kích thước dữ liệu gốc: 7043 dòng, 16 cột.



,ID,Churn,SeniorCitizen,Partner,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Tenure,Contract,PaymentMethod,PaperlessBilling,MonthlyCharges,TotalCharges
0,3668-QPYBK,1,No,0,No,DSL,1,1,0,0,2,1,Mailed check,1,53.85,108.15
1,9237-HQITU,1,No,0,Yes,Fiber optic,0,0,0,0,2,1,Electronic check,1,70.70,151.65
2,9305-CDSKC,1,No,0,Yes,Fiber optic,0,0,1,0,8,1,Electronic check,1,99.65,820.50
3,7892-POOKP,1,No,1,Yes,Fiber optic,0,0,1,1,28,1,Electronic check,1,104.80,3046.05
4,0280-XJGEX,1,No,0,Yes,Fiber optic,0,1,1,0,49,1,Bank transfer (automatic),1,103.70,5036.30


In [12]:
df_raw['Tenure'].unique()

array([ 2,  8, 28, 49, 10,  1, 47, 17,  5, 34, 11, 15, 18,  9,  7, 12, 25,
       68, 55, 37,  3, 27, 20,  4, 58, 53, 13,  6, 19, 59, 16, 52, 24, 32,
       38, 54, 43, 63, 21, 69, 22, 61, 60, 48, 40, 23, 39, 35, 56, 65, 33,
       30, 45, 46, 62, 70, 50, 44, 71, 26, 14, 41, 66, 64, 29, 42, 67, 51,
       31, 57, 36, 72,  0])

In [13]:
df_raw['PaperlessBilling'].unique()

array([1, 0])

In [14]:
df_raw['PaymentMethod'].unique()

array(['Mailed check', 'Electronic check', 'Bank transfer (automatic)',
       'Credit card (automatic)'], dtype=object)

In [18]:
def feature_engineering(df):
    """
    Hàm tạo các biến đặc trưng mới phục vụ mô hình dự đoán Churn.
    Đầu vào là DataFrame đã được làm sạch cơ bản.
    """

    df_features = df_raw.copy()
    # 1. tenure_group: Phân nhóm thời gian sử dụng

    bins = [0, 12, 24, 36, 48, 60, 72, np.inf]
    labels = ['0-12 months', '13-24 months', '25-36 months', '37-48 months', '49-60 months', '61-72 months', '72+ months']
    df_features['tenure_group'] = pd.cut(df_features['Tenure'], bins= bins,labels = labels, right = True)

    # 2. service_diversity: Tính tổng số dịch vụ phụ trợ đang sử dụng
    # Giả định các cột dịch vụ mang giá trị 1 (Yes) và 0 (No)

    services_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df_features['service_diversity'] = df_features[services_cols].sum(axis = 1)

    # 3. monthly_charges_ratio: Tỷ lệ chi phí tháng so với mức trung bình

    mean_monthly_charges = df_features['MonthlyCharges'].mean()
    df_features['monthly_charges_ratio'] = df_features['MonthlyCharges'] / mean_monthly_charges

    # 4. is_paperless_electronic: Tổ hợp hóa đơn điện tử và thanh toán tự động
    # Kiểm tra PaperlessBilling == 1 và PaymentMethod có chứa từ khóa 'automatic'
    is_paperless = df_features['PaperlessBilling'] == 1
    is_auto_payment = df_features['PaymentMethod'].str.contains('automatic',case = False, na = False)
    df_features['is_paperless_electronic'] = (is_paperless & is_auto_payment).astype(int)

    return df_features

df_cols = [['tenure_group', 'service_diversity', 'monthly_charges_ratio', 'is_paperless_electronic']]
df_text = feature_engineering(df_cols)
print(df_text[['tenure_group', 'service_diversity', 'monthly_charges_ratio', 'is_paperless_electronic']].head())
df_text.head()

   tenure_group  service_diversity  monthly_charges_ratio  \
0   0-12 months                  2               0.831510   
1   0-12 months                  0               1.091695   
2   0-12 months                  1               1.538718   
3  25-36 months                  2               1.618241   
4  49-60 months                  2               1.601255   

   is_paperless_electronic  
0                        0  
1                        0  
2                        0  
3                        0  
4                        1  


,ID,Churn,SeniorCitizen,Partner,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Tenure,Contract,PaymentMethod,PaperlessBilling,MonthlyCharges,TotalCharges,tenure_group,service_diversity,monthly_charges_ratio,is_paperless_electronic
0,3668-QPYBK,1,No,0,No,DSL,1,1,0,0,2,1,Mailed check,1,53.85,108.15,0-12 months,2,0.831510,0
1,9237-HQITU,1,No,0,Yes,Fiber optic,0,0,0,0,2,1,Electronic check,1,70.70,151.65,0-12 months,0,1.091695,0
2,9305-CDSKC,1,No,0,Yes,Fiber optic,0,0,1,0,8,1,Electronic check,1,99.65,820.50,0-12 months,1,1.538718,0
3,7892-POOKP,1,No,1,Yes,Fiber optic,0,0,1,1,28,1,Electronic check,1,104.80,3046.05,25-36 months,2,1.618241,0
4,0280-XJGEX,1,No,0,Yes,Fiber optic,0,1,1,0,49,1,Bank transfer (automatic),1,103.70,5036.30,49-60 months,2,1.601255,1
